In [ ]:
from collections.abc import Mapping
import dataclasses as dc
from typing import Generic, Self, Type, TypeAlias, TypeVar
import splatlog

K = TypeVar("K")
V = TypeVar("V")


@dc.dataclass
class Pair(Generic[K, V]):
    @classmethod
    def from_(cls: Type[Self], value: object) -> Self:
        if isinstance(value, cls):
            return value

        field_a, field_b = dc.fields(cls)

        match value:
            case (a, b) | [a, b] | {field_a.name: a, field_b.name: b}:
                if to_a := field_a.metadata.get("convert"):
                    a = to_a(a)

                if to_b := field_b.metadata.get("convert"):
                    b = to_b(b)

                return cls(a, b)  # type: ignore

        raise TypeError(
            "failed to construct `{}` from `{}`: `{!r}".format(
                cls, type(value), value
            )
        )

    @classmethod
    def collect(cls: Type[Self], value: object) -> dict[K, V]:
        m: dict[K, V] = {}

        if isinstance(value, Mapping):
            for kv in value.items():
                pair = cls.from_(kv)
                m[cls.key(pair)] = cls.value(pair)

        return m

    @classmethod
    def key(cls: Type[Self], instance: Self) -> K:
        return getattr(instance, dc.fields(cls)[0].name)

    @classmethod
    def value(cls: Type[Self], instance: Self) -> V:
        return getattr(instance, dc.fields(cls)[1].name)


@dc.dataclass
class VerbosityLevel(Pair[splatlog.Verbosity, splatlog.Level]):
    verbosity: splatlog.Verbosity = dc.field(
        metadata={"convert": splatlog.to_verbosity}
    )

    level: splatlog.Level = dc.field(metadata={"convert": splatlog.to_level})


f = VerbosityLevel.from_

assert f((123, 456)) == f([123, 456]) == f({"verbosity": 123, "level": 456})

assert f((0, splatlog.DEBUG)) == f((0, "debug"))

g = VerbosityLevel.collect

c = g({0: splatlog.WARNING, 1: "info", 3: "DEBUG"})
c

{0: 30, 1: 20, 3: 10}